In [1]:
# ==========================================
# FEATURE 2: AI ADVISOR ENGINE
# 06_ai_advisor_engine.ipynb
# ==========================================

import pandas as pd
import numpy as np
import joblib
import shap
import json
import warnings
from typing import Dict, Any, List, Optional
from scipy.stats import entropy, percentileofscore

# Suppress warnings for clean production output
warnings.filterwarnings('ignore')

print("✅ Environment initialized. Ready to build AI Advisor Engine.")

c:\Users\vishw\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment initialized. Ready to build AI Advisor Engine.


In [3]:
# ==========================================
# FEATURE EXPLAINABILITY MAPPING
# ==========================================

FEATURE_DESCRIPTIONS = {
    "total_clicks": "Overall volume of interaction with the virtual learning environment.",
    "active_days": "Number of unique days the student logged into the system.",
    "avg_daily_clicks": "Average number of interactions per active day.",
    "engagement_span_days": "Total days between the student's first and last interaction.",
    "activity_density": "Consistency of engagement over their active span.",
    "click_std": "Variability in day-to-day click behavior.",
    "daily_click_mean": "Mean clicks per day across the entire module duration.",
    "click_cv": "Coefficient of variation in daily clicks (consistency metric).",
    "unique_sites_visited": "Breadth of course material accessed by the student.",
    "clicks_early": "Engagement level during the first third of the module.",
    "clicks_mid": "Engagement level during the middle third of the module.",
    "clicks_late": "Engagement level during the final third of the module.",
    "early_engagement_ratio": "Proportion of total effort exerted early in the course.",
    "assessment_mean_score": "Average score across all submitted assessments.",
    "assessment_median_score": "Median score across all submitted assessments.",
    "assessment_max_score": "Highest score achieved on a single assessment.",
    "assessment_min_score": "Lowest score achieved on a single assessment.",
    "assessment_std_score": "Consistency of grades across different assessments.",
    "assessment_count": "Total number of assessments submitted.",
    "weighted_score_total": "Total accumulated points weighted by assessment importance.",
    "weighted_score_mean": "Average weighted score per assessment.",
    "avg_submission_day": "Average timing of assessment submissions relative to deadlines.",
    "first_submission_day": "Timing of the student's very first assessment submission.",
    "last_submission_day": "Timing of the student's final assessment submission.",
    "submission_day_std": "Variability in how early or late the student submits work."
}

In [4]:
# ==========================================
# CORE CLASS: AI Advisor Engine
# ==========================================

class AIAdvisorEngine:
    """
    AI-driven automated reporting engine for the Smart Student Analytics Platform.
    Generates comprehensive, interpretable, and JSON-serializable advisor reports 
    combining predictions, SHAP explainability, and peer benchmarking.
    """

    def __init__(self, data_path: str, rf_path: str, kmeans_path: str, le_path: str):
        """Loads models, datasets, and initializes the SHAP explainer."""
        self.df = pd.read_csv(data_path)
        self.rf_model = joblib.load(rf_path)
        self.kmeans_model = joblib.load(kmeans_path)
        self.label_encoder = joblib.load(le_path)
        
        # Enforce exact feature ordering expected by the model
        self.rf_features = list(self.rf_model.feature_names_in_)
        
        # Initialize SHAP TreeExplainer for fast, local feature explainability
        self.explainer = shap.TreeExplainer(self.rf_model)
        
    def _get_student_data(self, student_id: int) -> pd.Series:
        """Validates and retrieves student record."""
        student_data = self.df[self.df['id_student'] == student_id]
        if student_data.empty:
            raise ValueError(f"Student ID {student_id} not found in the dataset.")
        return student_data.iloc[0]

    def _align_features(self, student_series: pd.Series) -> pd.DataFrame:
        """Extracts and orders features exactly as the Random Forest requires."""
        return pd.DataFrame([student_series]).reindex(columns=self.rf_features, fill_value=0)

    def _compute_shap_explanations(self, aligned_df: pd.DataFrame, predicted_class_idx: int) -> Dict[str, Any]:
        """Calculates Local SHAP values to identify top positive and negative drivers."""
        shap_values = self.explainer.shap_values(aligned_df)
        
        # Handle SHAP multi-class format (list of arrays)
        if isinstance(shap_values, list):
            class_shap = shap_values[predicted_class_idx][0]
        else:
            class_shap = shap_values[0, :, predicted_class_idx]
            
        feature_impacts = {feat: float(val) for feat, val in zip(self.rf_features, class_shap)}
        sorted_impacts = sorted(feature_impacts.items(), key=lambda x: x[1], reverse=True)
        
        # Top 3 drivers pulling towards the predicted class
        top_pos = [{"feature": k, "impact": v, "explanation": FEATURE_DESCRIPTIONS.get(k, "Feature related to student activity.")} 
                   for k, v in sorted_impacts[:3] if v > 0]
                   
        # Top 3 drivers pulling away from the predicted class
        top_neg = [{"feature": k, "impact": v, "explanation": FEATURE_DESCRIPTIONS.get(k, "Feature related to student activity.")} 
                   for k, v in sorted_impacts[-3:] if v < 0]

        return {"top_positive_factors": top_pos, "top_negative_factors": top_neg}

    def _compute_peer_benchmark(self, student: pd.Series) -> List[Dict[str, Any]]:
        """Compares student to peers in the same module who passed/achieved distinction."""
        peers = self.df[(self.df['code_module'] == student['code_module']) & 
                        (self.df['final_result'].isin(['Pass', 'Distinction']))]
        
        metrics = ['total_clicks', 'active_days', 'assessment_mean_score', 'early_engagement_ratio']
        benchmark_results = []
        
        for m in metrics:
            s_val = float(student.get(m, 0))
            p_median = float(peers[m].median())
            percentile = float(percentileofscore(peers[m].dropna(), s_val))
            
            benchmark_results.append({
                "metric": m,
                "student_value": round(s_val, 2),
                "peer_median": round(p_median, 2),
                "difference": round(s_val - p_median, 2),
                "percentile_rank_among_successful_peers": round(percentile, 1)
            })
            
        return benchmark_results

    def _extract_interventions(self, student: pd.Series) -> List[str]:
        """Extracts prioritized interventions from the dataset."""
        interventions = []
        for i in range(1, 5):
            rec = student.get(f'recommendation_{i}')
            if pd.notna(rec) and str(rec).strip() not in ['', 'None', 'nan']:
                interventions.append(str(rec))
        return interventions

    def _calculate_confidence_metrics(self, aligned_df: pd.DataFrame) -> Dict[str, Any]:
        """Calculates prediction probabilities and entropy (uncertainty)."""
        probs = self.rf_model.predict_proba(aligned_df)[0]
        prob_dict = {str(c): round(float(p), 4) for c, p in zip(self.label_encoder.classes_, probs)}
        
        # Calculate Information Entropy (Lower entropy = Higher model certainty)
        pred_entropy = entropy(probs, base=2)
        confidence_score = float(max(probs) * 100)
        
        return {
            "class_probabilities": prob_dict,
            "prediction_entropy": round(float(pred_entropy), 4),
            "confidence_score_percent": round(confidence_score, 2)
        }

    def _generate_executive_summary(self, student: pd.Series, pred_label: str, 
                                    shap_data: Dict, interventions: List[str]) -> str:
        """Generates an NLP-style executive summary for university administrators."""
        risk = student.get('risk_level', 'Unknown')
        
        top_strength = shap_data['top_positive_factors'][0]['feature'].replace('_', ' ') if shap_data['top_positive_factors'] else "their general consistency"
        top_weakness = shap_data['top_negative_factors'][-1]['feature'].replace('_', ' ') if shap_data['top_negative_factors'] else "minor inconsistencies in engagement"
        
        first_action = interventions[0] if interventions else "continue standard monitoring"
        
        summary = (
            f"This student is currently classified at a '{risk}' risk level. "
            f"If no action is taken, the most likely academic outcome is '{pred_label}'. "
            f"Their strongest positive behavioral trait currently is {top_strength}, which is supporting their trajectory. "
            f"However, their biggest weakness pulling their performance down is related to {top_weakness}. "
            f"To optimize this student's outcome, the university should first prioritize: {first_action}."
        )
        return summary

    def generate(self, student_id: int) -> Dict[str, Any]:
        """
        Orchestrator method. Generates the full, JSON-ready advisor report.
        """
        # 1. Load and Align
        student = self._get_student_data(student_id)
        df_aligned = self._align_features(student)
        
        # 2. Predictions
        pred_idx = self.rf_model.predict(df_aligned)[0]
        pred_label = str(self.label_encoder.inverse_transform([pred_idx])[0])
        
        # 3. Sub-components
        confidence_metrics = self._calculate_confidence_metrics(df_aligned)
        shap_explanations = self._compute_shap_explanations(df_aligned, pred_idx)
        peer_benchmarks = self._compute_peer_benchmark(student)
        interventions = self._extract_interventions(student)
        
        # 4. Construct Final Report
        report = {
            "section_1_student_summary": {
                "student_id": int(student['id_student']),
                "module": str(student.get('code_module', 'Unknown')),
                "predicted_outcome": pred_label,
                "prediction_confidence_percent": confidence_metrics['confidence_score_percent'],
                "cluster_name": str(student.get('cluster_name', 'Unknown')),
                "risk_level": str(student.get('risk_level', 'Unknown'))
            },
            "section_2_academic_performance": {
                "assessment_mean_score": float(student.get('assessment_mean_score', 0)),
                "total_clicks": int(student.get('total_clicks', 0)),
                "active_days": int(student.get('active_days', 0)),
                "submission_behavior_std": float(student.get('submission_day_std', 0)),
                "behavior_risk_index": float(student.get('behavior_risk_index', 0)),
                "overall_status": "On Track" if pred_label in ['Pass', 'Distinction'] else "At Risk"
            },
            "section_3_explainability_shap": shap_explanations,
            "section_4_peer_benchmark": peer_benchmarks,
            "section_5_intervention_plan": {
                "priority_count": len(interventions),
                "recommended_actions": interventions
            },
            "section_6_executive_summary": self._generate_executive_summary(student, pred_label, shap_explanations, interventions),
            "section_7_confidence_metrics": confidence_metrics
        }
        
        return report

In [6]:
# ==========================================
# EXECUTION & TESTING
# ==========================================

# 1. Initialize the Engine using Absolute Paths
try:
    advisor = AIAdvisorEngine(
        data_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/data/processed/student_clustered.csv',
        rf_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_random_forest.pkl',
        kmeans_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_kmeans.pkl',
        le_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/label_encoder.pkl'
    )
    print("✅ AI Advisor Engine initialized successfully.")
    
except FileNotFoundError as e:
    print(f"❌ Initialization Error: Please ensure your file paths are correct. {e}")

# 2. Generate Report for a sample student
TARGET_STUDENT = 11391

try:
    # Only run if advisor was successfully created
    if 'advisor' in locals():
        report_dictionary = advisor.generate(student_id=TARGET_STUDENT)
        
        # 3. Output as formatted JSON
        print("\n" + "="*50)
        print(f"📄 AI ADVISOR REPORT FOR STUDENT {TARGET_STUDENT}")
        print("="*50)
        print(json.dumps(report_dictionary, indent=4))
    
except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during generation: {e}")

✅ AI Advisor Engine initialized successfully.

📄 AI ADVISOR REPORT FOR STUDENT 11391
{
    "section_1_student_summary": {
        "student_id": 11391,
        "module": "AAA",
        "predicted_outcome": "Pass",
        "prediction_confidence_percent": 84.65,
        "cluster_name": "Consistent Learners",
        "risk_level": "Low"
    },
    "section_2_academic_performance": {
        "assessment_mean_score": 82.0,
        "total_clicks": 922,
        "active_days": 40,
        "submission_behavior_std": 79.10309728449323,
        "behavior_risk_index": 0.0,
        "overall_status": "On Track"
    },
    "section_3_explainability_shap": {
        "top_positive_factors": [
            {
                "feature": "last_submission_day",
                "impact": 0.08589381582075667,
                "explanation": "Timing of the student's final assessment submission."
            },
            {
                "feature": "weighted_score_total",
                "impact": 0.0713460249